# SharePoint Automation Engine (Advanced Edition)
Automated configuration of Lists, Columns, Views (with filters), and Site Permission Levels.

## 1. Setup & Config Loader
Loading `sp_config.xlsx` and preparing the site environment.

In [ ]:
import pandas as pd
import json
import requests
import os
from typing import Dict, Any, List

CONFIG_PATH = 'sp_config.xlsx'

def load_config(path: str) -> Dict[str, pd.DataFrame]:
    xl = pd.ExcelFile(path)
    return {sheet: xl.parse(sheet) for sheet in xl.sheet_names}

configs = load_config(CONFIG_PATH)
settings = configs['Settings'].set_index('Key')['Value'].to_dict()

SITE_URL = settings.get('SITE_URL')
TENANT_ID = settings.get('TENANT_ID')
CLIENT_ID = settings.get('CLIENT_ID')
CLIENT_SECRET = settings.get('CLIENT_SECRET')

print(f"Config Loaded. Targeting: {SITE_URL}")

## 2. Authentication
Acquire tokens and Role Definition mappings.

In [ ]:
def get_access_token():
    return "YOUR_TOKEN_HERE"

TOKEN = get_access_token()
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json;odata=verbose",
    "Content-Type": "application/json;odata=verbose"
}

# Friendly name to Role Definition ID mapping (Standard IDs)
ROLE_MAP = {
    'Full Control': 1073741829,
    'Design': 1073741828,
    'Edit': 1073741830,
    'Contribute': 1073741827,
    'Read': 1073741826
}

## 3. Core CRUD Actions
Functions for Lists, Fields, Views, and Permissions.

In [ ]:
def manage_list(title, description, template, action):
    if action == 'Create':
        print(f"[LIST] Creating: {title}")
    elif action == 'Delete':
        print(f"[LIST] Deleting: {title}")

def manage_column(list_title, col_name, col_type, required, read_only, default_val, choices, action):
    print(f"[COLUMN] {action}: {col_name} on {list_title} (Type: {col_type})")

def manage_view(list_title, view_title, query, fields, row_limit, is_default, action):
    """Manage SharePoint Views with CAML/OData Filters."""
    print(f"[VIEW] {action}: {view_title} on {list_title}")
    if query:
        print(f"  -> Filter: {query}")
    print(f"  -> Visible Fields: {fields}")

def manage_roles(group_name, access_level, action):
    """Assign permission levels to site groups."""
    role_id = ROLE_MAP.get(access_level, ROLE_MAP['Read'])
    print(f"[ROLE] {action}: '{group_name}' as '{access_level}' (ID: {role_id})")

def manage_security(group_name, user_email, action):
    print(f"[USER] {action}: {user_email} -> {group_name}")

## 4. Automation Engine
Orchestrating the sequence of events.

In [ ]:
def run_automation():
    print("--- STARTING SHAREPOINT AUTOMATION ---")
    
    # 1. Security & Groups
    if 'Roles' in configs:
        for _, row in configs['Roles'].iterrows():
            manage_roles(row['GroupName'], row['AccessLevel'], row['Action'])
            
    # 2. Lists
    if 'Lists' in configs:
        for _, row in configs['Lists'].iterrows():
            manage_list(row['Title'], row['Description'], row['Template'], row['Action'])
            
    # 3. Columns
    if 'Columns' in configs:
        for _, row in configs['Columns'].iterrows():
            manage_column(row['ListTitle'], row['Name'], row['Type'], row['Required'], 
                          row['ReadOnly'], row['DefaultValue'], row['Choices'], row['Action'])
            
    # 4. Views
    if 'Views' in configs:
        for _, row in configs['Views'].iterrows():
            manage_view(row['ListTitle'], row['ViewTitle'], row['Query'], 
                        row['ViewFields'], row['RowLimit'], row['DefaultView'], row['Action'])

    # 5. User Assignments
    if 'Security' in configs:
        for _, row in configs['Security'].iterrows():
            manage_security(row['GroupName'], row['UserEmail'], row['Action'])

    print("--- AUTOMATION COMPLETE ---")

run_automation()